In [1]:
# Importing torch and setting the device
import torch

# Setting GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Properties
print(torch.cuda.get_device_name())
print(torch.cuda.get_device_capability())

NVIDIA GeForce MX350
(6, 1)


---

A PyTorch computational graph is a Directed Acyclic Graph (DAG) that represents the sequence of operations performed on tensors during the forward pass of a model. This graph is dynamically built during runtime, meaning it is constructed on the fly as operations are executed.

#### AutoGrad for Scalers

In [2]:
x = torch.tensor(data=3.0, requires_grad=True) # Enables the differentation wrt this variable (x)
x

tensor(3., requires_grad=True)

The differentiation is only possible when all the values in the tensor are `floating point` values and `not integer` values.

In [3]:
y = x ** 2
y

tensor(9., grad_fn=<PowBackward0>)

`backward` - Computes the gradient of current tensor wrt graph leaves. The graph is differentiated using the chain rule.

In [4]:
y.backward(retain_graph = True)
# Computational Graph: X -> square function -> y(leaf node)

In [5]:
x.grad
# Differentation of y wrt x

tensor(6.)

When you run `y.backward()` again (only if `retain_graph = True`) then PyTorch saves the gradient in its memory and add it when you run it again.

If you want to run `y.backward()` again, then its previous call must have `retain_graph = True` otherwise it will through a `RuntimeError: Trying to backward through the graph a second time`.

In [6]:
x = torch.tensor(4.0, requires_grad=True)
y = 2 * x + torch.cos(x) ** 2 # Addition Rule of Derivative
z = torch.sin(y)

x, y, z

(tensor(4., requires_grad=True),
 tensor(8.4272, grad_fn=<AddBackward0>),
 tensor(0.8401, grad_fn=<SinBackward0>))

PyTorch computes the graph based on the operations you perform. `MulBackward0` - Multiplication, `SinBackward0` - Sin(x). If certain equation have more than one operation e.g. ${2 * x + torch.cos(x) ^ 2}$ then the higher level operation (last operation) get mentioned in the gradient function i.e. `AddBackward`

Computational Graph: ${x → func(2 * x + cos(x) ^ 2) = y → func(sin(y)) = z}$

In [7]:
z.backward() # When you run the `backward` second time it will throw an error

In [8]:
x.grad
# y.grad - You only able to calculate the gradient wrt leaf node (the last variable)

tensor(-0.5482)

Gradients of non leaf node (y, z) will not be calculated

---

#### Basic AutoGrad from scratch

In [9]:
# Inputs
x = torch.tensor(6.7)  # Input feature
y = torch.tensor(0.0)  # True label (binary)

w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias

In [10]:
# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

In [11]:
# Forward pass
z = w * x + b  # Weighted sum (linear part)
y_pred = torch.sigmoid(z)  # Predicted probability

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)
loss

tensor(6.7012)

In [12]:
# Derivatives:
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x  # dz/dw = x
dz_db = 1  # dz/db = 1 (bias contributes directly to z)

# Chain Rule
dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [13]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


In [14]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)

w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

In [15]:
# Forward Pass
z = (w * x) + b
z 

tensor(6.7000, grad_fn=<AddBackward0>)

In [16]:
# Predict Probabilities
y_pred = torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [17]:
loss = binary_cross_entropy_loss(y_pred, y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

Computational Graph: ${w, b → foreard pass(x, w, b) = z → probabilities(z) = ypred → loss(ypred, yactual)}$

In [18]:
loss.backward()

print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


---

#### AutoGrad for Tensors

In [19]:
# Gradients wrt multiple X values
x = torch.tensor([[1.0, 2.0, 3.0], [2.5, 9.8, 5.7]], requires_grad = True)
x

tensor([[1.0000, 2.0000, 3.0000],
        [2.5000, 9.8000, 5.7000]], requires_grad=True)

In [20]:
y = (x ** 2).mean()
y

tensor(24.7967, grad_fn=<MeanBackward0>)

In [21]:
y.backward()
x.grad

tensor([[0.3333, 0.6667, 1.0000],
        [0.8333, 3.2667, 1.9000]])

A variable (y) on which you apply `backward` must be scaler and not tensor, because torch will only be able to calculate the gradient of scaler function wrt a tensor. if the last variable contains more than one values then torch gets confused about which value corrosponding to which gradient.

---

#### Gradient Accumulation

In [22]:
# clearing grad
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [23]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [24]:
y.backward(retain_graph=True)
print(f"Gradient after 1st backward cycle: {x.grad}", end="\n\n")

for i in range(4):
    y.backward(retain_graph=True)
    print(f"Gradient after {i + 2}th backward cycle: {x.grad}")

Gradient after 1st backward cycle: 4.0

Gradient after 2th backward cycle: 8.0
Gradient after 3th backward cycle: 12.0
Gradient after 4th backward cycle: 16.0
Gradient after 5th backward cycle: 20.0


You will not be able to run the `y.backward()` more than one time otherwise pytorch throws an error unless `retain_graph=True`

The problem with executing `y.backward()` and `x.grad` muptiple times will store the gradient values in the memory. If you run this more than one time then ouptut will be `d1 + d2 + ...` :  The values of gradients will get added only if `retain_graph=True`

In [25]:
x: torch.Tensor = torch.tensor(data=4, dtype=torch.float32, pin_memory=True, requires_grad=True)
x.grad = torch.tensor(data=0.0) # explicitely did this, because initially the value is None
y: torch.Tensor = x ** 2

x.grad.zero_() # Inplace Operation
y.backward(retain_graph=True) # Remove the previous gradient first (zero_grad), then calculates the gradient wrt current values.
print(f"Gradient after 1st backward cycle: {x.grad}", end="\n\n")

for i in range(4):
    x.grad = torch.zeros_like(input=x.grad)
    y.backward(retain_graph=True)
    print(f"Gradient after {i + 2}th backward cycle: {x.grad}")

Gradient after 1st backward cycle: 8.0

Gradient after 2th backward cycle: 8.0
Gradient after 3th backward cycle: 8.0
Gradient after 4th backward cycle: 8.0
Gradient after 5th backward cycle: 8.0


You dont need to do this manually, optimizers `zero_grad()` will do this for all the parameters

---

#### Disable Gradient Tracking

In [26]:
import torch
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2

y.backward(retain_graph=True)
x.grad

tensor(4.)

In [27]:
# Way 1: requires_grad_ - perminantly disable the gradients
x.requires_grad_(False) # requires_grad = False (_ inplace)

tensor(2.)

In [28]:
y = x ** 2 # Calculating y again after updating the state of x

try:
    y.backward()
except Exception as e:
    print(e)

element 0 of tensors does not require grad and does not have a grad_fn


In [29]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
z = y + 2

z.backward()
x.grad

tensor(4.)

You can only change `requires_grad` flags of leaf variables (starting veriables) : x.

In [30]:
# Way 2: detach() - The newly returned variable having same value gets detached from the DAG
y_ = y.detach() # Copy the value of x but its requires_grad becomes false
y_

tensor(4.)

In [31]:
z_ = y_ + 2
z_

tensor(6.)

In [32]:
try:
    z_.backward()
except Exception as e:
    print(e)

element 0 of tensors does not require grad and does not have a grad_fn


In [33]:
a = torch.tensor(data=[2, 3], dtype=torch.float32, requires_grad=True)
b = a.pow(2) + torch.sin(a)
c = b.sum() # Here, b is a part of DAG

b.detach_() # Why no error - Since you have calculated `c` before detaching `b`.

c.backward()
print(a.grad)

tensor([3.5839, 5.0100])


In [34]:
# Consider you are training your neural network
x = torch.tensor(2.4, requires_grad=True)
y = x ** 2

# Performing Gradient Optimization
y.backward()
print(x.grad) # The neural network is trained

# Its prediction time
# Way 3: Context Manager - torch.no_grad()
with torch.no_grad(): # context manager
    # Only in this scope, all the parameter's requires_grad parameter becomes false
    y = x + 10

    try:
        y.backward()
        print(x.grad())
    except Exception as e:
        print(e)

# For an instance, need to run a block of code without building a DAG, then you can stop the gradients by using context manager

tensor(4.8000)
element 0 of tensors does not require grad and does not have a grad_fn


`torch.no_grad()` is the most recommended way to stop calculating or tracking grad

---

[Gradients for non-differentiable functions](https://docs.pytorch.org/docs/stable/notes/autograd.html#gradients-for-non-differentiable-functions)

In [35]:
a = torch.tensor(data=[-1.0], requires_grad=True)
b = torch.sqrt(a)

b.backward()
print(a.grad)

tensor([nan])


In [36]:
a = torch.tensor(data=[-.0], requires_grad=True)
b = torch.sqrt(a)

b.backward()
print(a.grad)

tensor([-inf])


In [37]:
a = torch.tensor(data=0.0, requires_grad=True)
b = torch.log(a)

b.backward()
print(a.grad)

tensor(inf)


In [38]:
a = torch.tensor(data=-1.0, requires_grad=True)
b = torch.log(a)

b.backward()
print(a.grad)

tensor(-1.)


[Division by Zero in Autograd](https://docs.pytorch.org/docs/stable/notes/autograd.html#division-by-zero-in-autograd) 

In [39]:
input = torch.tensor([1., 1., 1., 1.], requires_grad=True)
div = torch.tensor([0., 1., 1., 0.])

mask = div != 0; print(mask)
safe_input = torch.zeros_like(input=input)
safe_input[mask] = input[mask] / div[mask]; print(safe_input)

loss = safe_input.sum()
loss.backward()

print(input.grad)

tensor([False,  True,  True, False])
tensor([0., 1., 1., 0.], grad_fn=<IndexPutBackward0>)
tensor([0., 1., 1., 0.])


Unresoved: instead of using `safe_input[mask] = input[mask] ÷ div[mask]` I've used `safe_input[mask] = (input ÷ div)[mask]`, it returned `tensor([nan, 1., 1., nan])`